# Modelo 4: Autoencoder con TensorFlow/Keras
Diseño, entrenamiento y definición del umbral de detección para el Autoencoder.

In [1]:
# CELDA 1: Cargar librerías y datos escalados
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import roc_auc_score, classification_report
import joblib
import os

X_train_sc = np.load('../data/processed/X_train_sc.npy')
X_test_sc  = np.load('../data/processed/X_test_sc.npy')
y_train    = np.load('../data/processed/y_train.npy')
y_test     = np.load('../data/processed/y_test.npy')

# Solo entrenamos con transacciones legítimas (y_train == 0)
X_train_legitimas = X_train_sc[y_train == 0]
n_features = X_train_sc.shape[1]

print('Legitimate train size:', X_train_legitimas.shape)
print('Number of features:', n_features)

Legitimate train size: (301414, 12)
Number of features: 12


In [2]:
# CELDA 2: Arquitectura del Autoencoder
tf.random.set_seed(42)
entrada = keras.Input(shape=(n_features,))
# Encoder (compresión)
cod = keras.layers.Dense(32, activation='relu')(entrada)
cod = keras.layers.Dense(16, activation='relu')(cod)
cod = keras.layers.Dense(8,  activation='relu')(cod)  # capa bottleneck
# Decoder (reconstrucción)
dec = keras.layers.Dense(16, activation='relu')(cod)
dec = keras.layers.Dense(32, activation='relu')(dec)
salida = keras.layers.Dense(n_features, activation='sigmoid')(dec)

autoencoder = keras.Model(inputs=entrada, outputs=salida)
autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.summary()

Model: "model"


_________________________________________________________________


 Layer (type)                Output Shape              Param #   


 input_1 (InputLayer)        [(None, 12)]              0         


 dense (Dense)               (None, 32)                416       


 dense_1 (Dense)             (None, 16)                528       


... (4 outputs left out)

Total params: 2164 (8.45 KB)


Trainable params: 2164 (8.45 KB)


Non-trainable params: 0 (0.00 Byte)


_________________________________________________________________


In [3]:
# CELDA 3: Entrenar el Autoencoder
historial = autoencoder.fit(
    X_train_legitimas, X_train_legitimas,
    epochs=50,
    batch_size=256,
    validation_split=0.1,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=5, restore_best_weights=True)
    ],
    verbose=1
)

Epoch 1/50


   1/1060 [..............................] - ETA: 11:22 - loss: 0.7434

  49/1060 [>.............................] - ETA: 1s - loss: 0.8585   

 101/1060 [=>............................] - ETA: 0s - loss: 0.7998

 158/1060 [===>..........................] - ETA: 0s - loss: 0.7162

 215/1060 [=====>........................] - ETA: 0s - loss: 0.7066

... (1,074 outputs left out)

 901/1060 [========================>.....] - ETA: 0s - loss: 0.5373

 955/1060 [==========================>...] - ETA: 0s - loss: 0.5448

1009/1060 [===========================>..] - ETA: 0s - loss: 0.5466

1060/1060 [==============================] - 1s 976us/step - loss: 0.5432 - val_loss: 0.5178


In [4]:
# CELDA 4: Definir el umbral de reconstrucción y evaluar preliminarmente
X_test_recon = autoencoder.predict(X_test_sc)
errores = np.mean(np.power(X_test_sc - X_test_recon, 2), axis=1)

# Umbral: percentil 95 del error en transacciones legítimas de test
errores_legitimas = errores[y_test == 0]
umbral = np.percentile(errores_legitimas, 95)
print(f'Umbral de detección: {umbral:.6f}')

y_pred_ae = (errores > umbral).astype(int)
print(classification_report(y_test, y_pred_ae, target_names=['Legítima','Fraude']))
print(f'AUC-ROC: {roc_auc_score(y_test, errores):.4f}')

   1/2057 [..............................] - ETA: 2:15

  79/2057 [>.............................] - ETA: 1s  

 174/2057 [=>............................] - ETA: 1s

 272/2057 [==>...........................] - ETA: 0s

 372/2057 [====>.........................] - ETA: 0s

 471/2057 [=====>........................] - ETA: 0s

... (14 outputs left out)

1917/2057 [==========================>...] - ETA: 0s

2013/2057 [============================>.] - ETA: 0s

2057/2057 [==============================] - 1s 525us/step


Umbral de detección: 0.952251
              precision    recall  f1-score   support

    Legítima       0.99      0.95      0.97     64592
      Fraude       0.10      0.30      0.15      1232

    accuracy                           0.94     65824
   macro avg       0.54      0.62      0.56     65824
weighted avg       0.97      0.94      0.95     65824

AUC-ROC: 0.7936


In [5]:
# CELDA 5: Guardar el modelo y su umbral
os.makedirs('../models', exist_ok=True)
autoencoder.save('../models/autoencoder.h5')
joblib.dump({'umbral': umbral}, '../models/autoencoder_config.pkl')
print('Autoencoder y configuración guardados.')

Autoencoder y configuración guardados.


C:\Users\oguis\Desktop\AIFraudesBancarios\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
